In [7]:
import pandas as pd
import numpy as np

# Cleaning CGGA/TCGA RNA-seq Expression Matrix

file_path = "CGGA_RNAseq.txt"
expression_df = pd.read_csv("CGGA_data/cleaned_RNAseq_data.csv", sep=",")

print(f"Original matrix shape: {expression_df.shape}")
print(f"Number of genes: {expression_df.shape[0]}")
print(f"Number of samples: {expression_df.shape[1] - 1}")

missing_indicators = [
    'not reported', 'unknown', 'not available', 'N/A', 'NA', '--', 'null', ' '
]
expression_df.replace(missing_indicators, np.nan, inplace=True)

null_count = expression_df.isnull().sum().sum()
print(f"\nTotal missing values found: {null_count}")

print(expression_df.columns)

first_col = expression_df.columns[0]


duplicate_genes = expression_df[first_col].duplicated().sum()
if duplicate_genes > 0:
    print(f"Found {duplicate_genes} duplicate gene names. Keeping the first occurrence.")
    expression_df = expression_df.drop_duplicates(subset=[first_col], keep='first')

expression_df.set_index(first_col, inplace=True)

threshold = 0.1 * len(expression_df.columns)
expression_df = expression_df[(expression_df > 0).sum(axis=1) >= threshold]

print(f"Shape after filtering low-expression genes: {expression_df.shape}")

expression_df = expression_df.apply(pd.to_numeric, errors='coerce')

output_path = 'CGGA_data/cleaned_RNAseq_data.csv'
expression_df.to_csv(output_path)

print(f"\nFinal cleaned shape: {expression_df.shape}")
print(f"Cleaned data saved to: {output_path}")

Original matrix shape: (19091, 703)
Number of genes: 19091
Number of samples: 702

Total missing values found: 0
Index(['sample', 'TCGA-06-0178-01', 'TCGA-02-2483-01', 'TCGA-06-5417-01',
       'TCGA-15-1444-01', 'TCGA-06-2570-01', 'TCGA-19-2629-01',
       'TCGA-27-2521-01', 'TCGA-26-1442-01', 'TCGA-06-0129-01',
       ...
       'TCGA-DU-7304-02', 'TCGA-DU-5872-02', 'TCGA-FG-A4MT-02',
       'TCGA-DU-6404-02', 'TCGA-TQ-A7RK-02', 'TCGA-DU-5870-02',
       'TCGA-TM-A7CF-02', 'TCGA-DU-6397-02', 'TCGA-TQ-A7RV-02',
       'TCGA-DH-A669-02'],
      dtype='object', length=703)
Shape after filtering low-expression genes: (19091, 702)

Final cleaned shape: (19091, 702)
Cleaned data saved to: CGGA_data/cleaned_RNAseq_data.csv


In [21]:
import pandas as pd

# Peek at the raw file to see what separator is used
with open("CGGA_data/combined_clinical_RNAseq.csv") as f:
    print(repr(f.readline()))  # shows \t vs , vs ; etc.

'combined_clinical_RNAseq\n'


In [25]:
import os
for f in os.listdir("CGGA_data/"):
    print(f)

cleaned_RNAseq_data.csv
.DS_Store
combined_clinical_RNAseq.csv
clean_cgga_clinical_data.csv
CGGA_RNAseq.txt
CGGA_clinical.txt


In [26]:
import pandas as pd

# Load originals
clinical = pd.read_csv("CGGA_data/CGGA_clinical.txt", sep="\t")
rnaseq = pd.read_csv("CGGA_data/CGGA_RNAseq.txt", sep="\t")

print("Clinical shape:", clinical.shape)
print("Clinical columns:", clinical.columns[:8].tolist())
print("\nRNAseq shape:", rnaseq.shape)
print("RNAseq columns:", rnaseq.columns[:5].tolist())

Clinical shape: (702, 9)
Clinical columns: ['ID', 'Histology', 'Grade', 'Gender', 'Age', 'OS', 'Censor', 'IDH_mutation_status']

RNAseq shape: (20530, 703)
RNAseq columns: ['sample', 'TCGA-06-0178-01', 'TCGA-02-2483-01', 'TCGA-06-5417-01', 'TCGA-15-1444-01']


In [29]:
import pandas as pd

# Load
clinical = pd.read_csv("CGGA_data/CGGA_clinical.txt", sep="\t")
rnaseq = pd.read_csv("CGGA_data/CGGA_RNAseq.txt", sep="\t")

# Transpose RNAseq: make samples rows, genes columns
rnaseq_t = rnaseq.set_index("sample").T
rnaseq_t.index.name = "ID"
rnaseq_t = rnaseq_t.reset_index()

print("Transposed RNAseq shape:", rnaseq_t.shape)  # should be (702, 20531)

# Filter clinical for GBM first
clinical_gbm = clinical[clinical["Histology"] == "GBM"]
print("GBM samples:", len(clinical_gbm))

# Merge on ID
combined = clinical_gbm.merge(rnaseq_t, on="ID", how="inner")
print("Combined shape:", combined.shape)

# Save
combined.to_csv("CGGA_data/combined_clinical_RNAseq.csv", index=False)
print("Done!")
combined.iloc[:, :15].head(20).style.set_properties(**{'font-size': '11px'})

Transposed RNAseq shape: (702, 20531)
GBM samples: 152
Combined shape: (152, 20539)
Done!


,ID,Histology,Grade,Gender,Age,OS,Censor,IDH_mutation_status,1p19q_codeletion_status,ARHGEF10L,HIF3A,RNF17,RNF10,RNF11,RNF13
0,TCGA-06-0178-01,GBM,Grade IV,Male,38.000000,1617.000000,0.000000,Mutant,Non-codel,9.886600,5.776500,0.000000,11.107600,11.037600,12.575500
1,TCGA-02-2483-01,GBM,Grade IV,Male,43.000000,459.000000,0.000000,Mutant,Non-codel,9.873800,4.953000,0.000000,11.658500,11.936800,11.190600
2,TCGA-06-5417-01,GBM,Grade IV,Female,45.000000,153.000000,0.000000,Mutant,nan,10.584400,9.274500,0.000000,11.689400,11.978500,11.530000
3,TCGA-15-1444-01,GBM,Grade IV,Male,21.000000,1515.000000,1.000000,Mutant,Non-codel,10.906100,6.890200,0.000000,11.490400,11.923900,11.914400
4,TCGA-06-2570-01,GBM,Grade IV,Female,21.000000,282.000000,0.000000,Mutant,Non-codel,10.928000,10.136900,0.000000,11.705400,11.077200,11.123500
5,TCGA-19-2629-01,GBM,Grade IV,Male,60.000000,726.000000,1.000000,Mutant,Non-codel,10.699200,6.417800,0.000000,11.642100,11.440300,11.672900
6,TCGA-27-2521-01,GBM,Grade IV,Male,34.000000,312.000000,0.000000,Mutant,Non-codel,11.036100,8.281500,0.000000,12.035800,11.210400,11.882400
7,TCGA-26-1442-01,GBM,Grade IV,Male,43.000000,939.000000,0.000000,Mutant,Non-codel,12.162300,9.770800,0.000000,11.829600,11.734000,11.305300
8,TCGA-06-0129-01,GBM,Grade IV,Male,30.000000,1008.000000,1.000000,Mutant,Non-codel,11.188900,11.035900,0.000000,11.465600,10.921300,11.242100
9,TCGA-19-1787-01,GBM,Grade IV,Male,48.000000,378.000000,1.000000,nan,Non-codel,10.693500,8.969900,0.000000,11.844200,11.587600,10.588500


In [24]:
df = pd.read_csv("CGGA_data/combined_clinical_RNAseq.csv", sep="\n")
print(df.shape)
print(df.columns[:5].tolist())

ValueError: Specified \n as separator or delimiter. This forces the python engine which does not accept a line terminator. Hence it is not allowed to use the line terminator as separator.

In [20]:
import pandas as pd

# No skiprows needed — headers are on the first row
df = pd.read_csv("CGGA_data/combined_clinical_RNAseq.csv", sep=",")

print(df.columns.tolist()[:10])  # sanity check
print(df.shape)

# Filter for GBM
df = df[df['Histology'] == 'GBM']

# Save
df.to_csv('CGGA_data/combined_clinical_RNAseq.csv', index=False)
print(f"Success! Final shape: {df.shape}")

['combined_clinical_RNAseq']
(703, 1)


KeyError: 'Histology'